# Build index tầng 2 và kiểm tra bộ nhớ, tìm kiếm (có / không rerank)

Notebook **chỉ gọi code của dự án** — cùng đường chạy với `src/pipeline.py` và `eval/evaluate.py`:

- văn bản nạp vào: `corpus_dir`, `corpus_files`, `ten_van_ban` trong `src/config.py`;
- đọc tầng 2 thành Node: `src/processed_loader.py`; build / nạp index: `src/index_builder.build_or_load_index`
  (index ở `storage/<profile>/...`, tự build lại khi văn bản hoặc model đổi);
- tìm kiếm: `src/retriever.search`; chấm theo (văn bản, Điều): `eval/evaluate.py`.

Phần riêng của notebook: đo **RAM, VRAM** ở từng mốc, dung lượng index, thời gian tìm. Chưa dùng LLM.
Rerank cần profile `server`.

In [1]:
import os, sys, time, logging, subprocess, dataclasses
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "eval"))

os.environ.setdefault("RAG_PROFILE", "server")   # "local" hoặc "server"
logging.getLogger("httpx").setLevel(logging.WARNING)

import psutil
import torch
from config import get_config

cfg = get_config()
REBUILD = False   # True: build lại dù index đã khớp corpus

_proc = psutil.Process()
MOC: list[tuple[str, float, float, float]] = []   # (mốc, RAM MiB, VRAM torch MiB, VRAM nvidia-smi MiB)


def vram_nvidia_smi() -> float:
    """VRAM tiến trình này chiếm theo nvidia-smi (gồm cả CUDA context), MiB; 0 nếu không có GPU."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10).stdout
    except (OSError, subprocess.TimeoutExpired):
        return 0.0
    for line in out.splitlines():
        pid, _, used = line.partition(",")
        if pid.strip() == str(_proc.pid):
            return float(used)
    return 0.0


def do_bo_nho(moc: str) -> None:
    ram = _proc.memory_info().rss / 2**20
    vram = torch.cuda.memory_allocated() / 2**20 if torch.cuda.is_available() else 0.0
    smi = vram_nvidia_smi()
    MOC.append((moc, ram, vram, smi))
    print(f"[bộ nhớ] {moc}: RAM {ram:,.0f} MiB | VRAM torch {vram:,.0f} MiB | VRAM nvidia-smi {smi:,.0f} MiB")


print("Profile:", cfg.name, "| embedding:", cfg.embed_model_name, "| reranker:", cfg.reranker_model_name)
print("Corpus :", cfg.corpus_dir)
print("Index  :", cfg.persist_dir)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có")
do_bo_nho("sau khi import")

Profile: server | embedding: AITeamVN/Vietnamese_Embedding | reranker: AITeamVN/Vietnamese_Reranker
Corpus : data/processed/01_quy_che_dao_tao/1_quy_che_dao_tao_dai_hoc
Index  : storage/server/01_quy_che_dao_tao/1_quy_che_dao_tao_dai_hoc
GPU    : NVIDIA RTX A2000 Laptop GPU
[bộ nhớ] sau khi import: RAM 545 MiB | VRAM torch 0 MiB | VRAM nvidia-smi 0 MiB


## 1. Văn bản nạp vào index

Danh sách lấy từ `cfg.corpus_files` / `cfg.ten_van_ban` (sửa trong `src/config.py`). Muốn thử nạp hết thư mục:
`cfg = dataclasses.replace(cfg, corpus_files=())` — index sẽ tự build lại vì corpus đổi.

In [2]:
from collections import Counter
from llama_index.core.schema import MetadataMode
from processed_loader import corpus_files, load_corpus_nodes, ten_hien_thi

for p in corpus_files(cfg):
    print(f"  - {ten_hien_thi(cfg, p) or '(chưa đặt tên)':58} {p.name}")

nodes = load_corpus_nodes(cfg)
so_tu = sorted(len(n.get_content().split()) for n in nodes)
print(f"\n{len(nodes)} Node | số từ/Node: min {so_tu[0]}, trung vị {so_tu[len(so_tu) // 2]}, max {so_tu[-1]}")
for (vb, phan), c in sorted(Counter((str(n.metadata["van_ban"]), str(n.metadata["phan"])) for n in nodes).items()):
    print(f"{c:4}  {phan:9} {vb}")

print("\nVí dụ văn bản đưa vào embedding (có ghép metadata):")
print(nodes[min(5, len(nodes) - 1)].get_content(metadata_mode=MetadataMode.EMBED)[:400])
do_bo_nho("sau khi đọc Node")

  - QĐ 2375/2022 về chuyển chương trình, ngành đào tạo         2022-2375-QyD Chuyen Chuong Trinh Dao Tao DoiVoiSinhVenDHCQ.md
  - Quy định khối lượng đào tạo hệ chính quy theo tín chỉ      22_quy-dinh-ve-khoi-luong-dao-tao-cua-cac-he-dao-tao-chinh-quy-dao-tao.md
  - Quy chế đào tạo trình độ đại học SGU 2021                  25_quy-che-dao-tao-trinh-do-dai-hoc-tai-truong-dai-hoc-sai-gon.md
  - QĐ 810 về học cùng lúc hai chương trình                    8_QD810QuyDinhVeHocCungLucHaiChuongTrinhDoiVoiSVDHCQ.md
  - QĐ 3183/2025 sửa đổi Quy chế đào tạo SGU 2021              QD 3183.2025 - Sua doi Quy che Dao tao trinh do DH_LienQuanRutMonHoc.md
  - Quy định số tín chỉ tối thiểu kỹ sư CNTT (khóa 2017 trở đi) Quydinh150tc.md

54 Node | số từ/Node: min 11, trung vị 185, max 720
  21  chinh     Quy chế đào tạo trình độ đại học SGU 2021
   1  noi_dung  Quy định khối lượng đào tạo hệ chính quy theo tín chỉ
   1  noi_dung  Quy định số tín chỉ tối thiểu kỹ sư CNTT (khóa 2017 trở đi)
   4  ban_hanh  Q

## 2. Build hoặc nạp lại index

`build_or_load_index` nạp model embedding (theo profile, fp16 nếu bật) rồi nạp index đã lưu nếu khớp corpus,
ngược lại build mới và lưu kèm `corpus.json`.

In [ ]:
from index_builder import build_or_load_index

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()
index = build_or_load_index(cfg, rebuild=REBUILD)
print(f"Nạp model + index: {time.time() - t0:.1f}s | {len(index.docstore.docs)} Node")

persist = ROOT / cfg.persist_dir
for f in sorted(persist.glob("*.json")):
    print(f"  {f.name:28} {f.stat().st_size / 2**10:8,.0f} KiB")
DISK_KIB = sum(f.stat().st_size for f in persist.glob("*.json")) / 2**10
print(f"  {'tổng':28} {DISK_KIB:8,.0f} KiB")
do_bo_nho("sau khi nạp model + index")

## 3. Tìm kiếm vector (chưa rerank)

`eval/evaluate.chay` chạy cả 2 bộ câu hỏi, chấm theo (văn bản, Điều), in câu ngoài top-3 và thời gian mỗi câu.

In [ ]:
from evaluate import chay, in_so_sanh  # pyright: ignore[reportMissingImports]  # eval/ thêm vào sys.path ở cell đầu
from retriever import get_postprocessors, node_label, search

ket_qua_vector = chay(lambda q: search(index, cfg, q, postprocessors=[]), (1, 3, 5, 10))
do_bo_nho("sau khi tìm vector 41 câu")

## 4. Tìm kiếm vector + rerank

Reranker nạp bằng `get_postprocessors()` (cùng cách `pipeline.py` dùng khi `use_rerank=True`), giữ `reranker_top_n` Node.

In [ ]:
cfg_rr = dataclasses.replace(cfg, use_rerank=True)
t0 = time.time()
rerankers = get_postprocessors(cfg_rr)
if not rerankers:
    raise RuntimeError(f"Profile {cfg.name!r} không có reranker — đặt RAG_PROFILE=server ở cell đầu")
print(f"Nạp reranker {cfg.reranker_model_name}: {time.time() - t0:.1f}s (giữ top-{cfg.reranker_top_n})")
do_bo_nho("sau khi nạp reranker")

ket_qua_rerank = chay(lambda q: search(index, cfg_rr, q, postprocessors=rerankers), (1, 3))
in_so_sanh(ket_qua_vector, ket_qua_rerank)
do_bo_nho("sau khi tìm có rerank 41 câu")

## 5. Thử tìm một câu bất kỳ (so sánh có / không rerank)

In [ ]:
def thu(cau_hoi: str, k: int = 5) -> None:
    print("Câu hỏi:", cau_hoi)
    for ten, res in (("vector", search(index, cfg, cau_hoi, postprocessors=[])[:k]),
                     ("rerank", search(index, cfg_rr, cau_hoi, postprocessors=rerankers))):
        print(f"  -- {ten}")
        for i, n in enumerate(res, 1):
            print(f"  [{i}] {n.score or 0:.3f} | {node_label(n)} | {n.node.get_content()[:60]!r}")
    print()


thu("Điểm C+ tương ứng với khoảng điểm nào trên thang điểm 10?")
thu("Sinh viên được rút học phần trong bao lâu?")

## 6. Tổng kết bộ nhớ

- **RAM**: bộ nhớ thường trú (RSS) của tiến trình kernel.
- **VRAM torch**: phần tensor PyTorch cấp phát (trọng số embedding + reranker + dữ liệu tạm).
- **VRAM nvidia-smi**: VRAM tiến trình chiếm trên card, gồm cả CUDA context — gần số thực tế để so với dung lượng card.

In [ ]:
print(f"{'Mốc':32} {'RAM':>9} {'Δ RAM':>8} {'VRAM torch':>11} {'VRAM smi':>9}")
prev = MOC[0][1]
for moc, ram, vram, smi in MOC:
    print(f"{moc:32} {ram:7,.0f} M {ram - prev:+7,.0f} {vram:9,.0f} M {smi:7,.0f} M")
    prev = ram
if torch.cuda.is_available():
    print(f"\nVRAM torch đỉnh: {torch.cuda.max_memory_allocated() / 2**20:,.0f} MiB"
          f" / card {torch.cuda.get_device_properties(0).total_memory / 2**20:,.0f} MiB")
print(f"Index trên đĩa: {DISK_KIB:,.0f} KiB cho {len(nodes)} Node")